# 06 Summary Report

In [ ]:
# User-editable papermill/environment configuration
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/tmp/wwpgd_v2/data"))
RESULTS_ROOT = Path(
    os.environ.get(
        "WWGPT_RESULTS_ROOT",
        os.environ.get(
            "RESULTS_ROOT",
            "/tmp/wwpgd_v2/real_level0_results_v5/"
            "experiments/level_00/multiplier_20",
        ),
    )
)
RUN_LOG = Path(os.environ.get("RUN_LOG", "/tmp/wwpgd_v2/real_level0_five_seed_v4.log"))
PID_FILE = Path(os.environ.get("PID_FILE", "/tmp/wwpgd_v2/real_level0_five_seed_v4.pid"))

ANALYSIS_DIR = RESULTS_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_LEGACY = False
EXPECTED_SEEDS = {1337, 2027, 4099, 7919, 104729}


In [ ]:
import sys
from pathlib import Path
cwd = Path.cwd().resolve()
repo = cwd if (cwd/'src'/'wwgpt').exists() else cwd.parent
if str(repo/'src') not in sys.path:
    sys.path.insert(0, str(repo/'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from wwgpt.analysis import *
RESULTS_ROOT = resolve_experiment_root(RESULTS_ROOT)
ANALYSIS_DIR = RESULTS_ROOT / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Run log: {RUN_LOG}')
print(f'PID file: {PID_FILE}')


Compact recomputed summary report for canonical scientific pairs.

In [ ]:
candidates = discover_pair_candidates(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
canonical_pairs, pair_audit = select_canonical_pairs(candidates)
runs = discover_canonical_runs(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
inventory = build_run_inventory(runs)
pair_audit.to_csv(ANALYSIS_DIR/'pair_candidate_audit.csv', index=False)
inventory.to_csv(ANALYSIS_DIR/'run_inventory.csv', index=False)
print(f'Canonical pairs: {len(canonical_pairs)}')
display(pair_audit)
display(inventory)


In [ ]:
term=terminal_results(runs); term.to_csv(ANALYSIS_DIR/'summary_paired_terminal_results.csv',index=False)
summary_diff=student_t_summary(term['wwpgd_minus_adamw_validation_loss']) if not term.empty else {}
inventory.to_csv(ANALYSIS_DIR/'summary_optimizer_metrics.csv',index=False)
projs=[]; specs=[]
for r in runs:
    art=load_run_artifacts(Path(r['run_dir']));
    if not art['projection'].empty: projs.append(art['projection'].assign(seed=r['seed']))
    if not art['spectral'].empty: specs.append(art['spectral'].assign(seed=r['seed'],optimizer_family=r['optimizer_family']))
proj=pd.concat(projs,ignore_index=True) if projs else pd.DataFrame(); proj.to_csv(ANALYSIS_DIR/'summary_projection_metrics.csv',index=False)
spec=pd.concat(specs,ignore_index=True) if specs else pd.DataFrame();
spec_summary=spec[spec.get('layer_group',pd.Series(dtype=str)).eq('projected_transformer_matrix')].groupby(['optimizer_family','tokens_seen']).agg(median_alpha=('alpha','median'),q25=('alpha',lambda s:s.quantile(.25)),q75=('alpha',lambda s:s.quantile(.75))).reset_index() if not spec.empty else pd.DataFrame(); spec_summary.to_csv(ANALYSIS_DIR/'summary_spectral_metrics.csv',index=False)
scaling_ready=scaling_readiness(scaling_design_points(discover_scaling_runs(RESULTS_ROOT.parent.parent.parent if RESULTS_ROOT.exists() else RESULTS_ROOT)))
report={'results_root':str(RESULTS_ROOT),'canonical_pair_count':len(canonical_pairs),'selected_seeds':[c.seed for c in canonical_pairs],'excluded_pair_count':int((pair_audit.status=='excluded').sum()),'schema_valid':bool((inventory.scientific_schema_version.astype(float)>=2).all()) if not inventory.empty else False,'paired_terminal_difference':summary_diff,'wwpgd_wins':int((term.get('wwpgd_minus_adamw_validation_loss',pd.Series(dtype=float))<0).sum()),'adamw_wins':int((term.get('wwpgd_minus_adamw_validation_loss',pd.Series(dtype=float))>0).sum()),'ties':int((term.get('wwpgd_minus_adamw_validation_loss',pd.Series(dtype=float))==0).sum()),'projection_event_count':int(proj.projection_event.nunique()) if not proj.empty else 0,'weightwatcher_valid':bool((spec.spectral_estimator.astype(str).str.lower()=='weightwatcher').all()) if not spec.empty else False,'scaling_readiness':scaling_ready.to_dict('records'),'methodological_limitation':'This is a five-seed comparison at one model size and one token budget. It can estimate a paired optimizer effect at this operating point, but it cannot establish how the effect scales with model size or compute.'}
import json
(ANALYSIS_DIR/'summary_report.json').write_text(json.dumps(report,indent=2,default=str))
lines = [
    '# Summary Report', '',
    f'Results root: `{RESULTS_ROOT}`', '',
    f'Selected canonical pairs: {len(canonical_pairs)}', '',
    f'Selected seeds: {[c.seed for c in canonical_pairs]}', '',
    f'Excluded duplicate/failed pairs: {int((pair_audit.status == "excluded").sum())}', '',
    f'Schema valid: {report["schema_valid"]}', '',
    f'Paired terminal validation-loss difference (WW-PGD minus AdamW): {summary_diff}', '',
    f'Projection events: {report["projection_event_count"]}', '',
    f'WeightWatcher validity: {report["weightwatcher_valid"]}', '',
    f'Scaling readiness: {scaling_ready.reason.iloc[0]}', '',
    'Important missing logging fields are listed in `logging_coverage.csv` when notebook 02 is run.', '',
    '## Methodological limitations', '',
    report['methodological_limitation'],
]
md = chr(10).join(lines) + chr(10)
(ANALYSIS_DIR/'summary_report.md').write_text(md)
print(md)
